# 🐖 04 | Segmentación y evaluación independiente

**RECURSO REQUERIDO: GPU (T4 o superior)**. Este notebook realiza exclusivamente su propia etapa. **No cambies CPU/GPU dentro del notebook.**

Proyecto: Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖. Datos: FaroPigSeg, licencia reportada CC BY-NC 4.0; citar fuente https://github.com/yudong888/FaroPig-Datasets.

**Ruta general:** `MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖/`. Ejecutar en orden 00 → 01 → 02 → 03 → 04. Todos los resultados se guardan en Drive.

## Objetivo
Usar `best.pt` ya entrenado para generar máscaras predichas, contar objetos visibles y evaluar **una sola vez** las métricas en el test agrupado por fecha. Guarda archivos Excel, PNG, JSON y un ZIP. No reentrena el modelo ni vuelve a preparar el dataset.

**ATENCIÓN:** las áreas obtenidas se expresan en píxeles. Este dataset es de imágenes fijas, no permite medir trayectorias ni comportamientos temporales.

In [ ]:
import torch
assert torch.cuda.is_available(),'Activa GPU T4 en Colab antes de ejecutar este notebook.'
print('✅ GPU:',torch.cuda.get_device_name(0))

In [ ]:
%pip -q install "ultralytics>=8.3,<9" pandas openpyxl pyyaml pillow

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import json, os

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖')
if not ROOT.is_dir():
    raise FileNotFoundError(
        'No se encontró la carpeta principal. Verifica la cuenta de Google y el nombre de la carpeta: '
        'Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖'
    )
DATA_ROOT = ROOT / '01_Datasets' / 'FaroPigSeg' / 'extraido' / 'FaroPigSeg'
DATASET_DIR = ROOT / '01_Datasets' / 'FaroPigSeg'
RES = ROOT / '03_Resultados'
EXPO = ROOT / '05_Exportaciones_ZIP'
for sub in ('01_Exploracion','02_Segmentacion','03_Metricas','04_Figuras','05_Reportes','06_Entrenamiento'):
    (RES / sub).mkdir(parents=True, exist_ok=True)
EXPO.mkdir(parents=True,exist_ok=True)
print('Proyecto:',ROOT)

In [ ]:
from ultralytics import YOLO
import pandas as pd, numpy as np, torch, yaml, cv2, zipfile, shutil, hashlib
YAML=DATASET_DIR/'split_fecha_v1'/'data_fecha.yaml'
BEST=RES/'06_Entrenamiento'/'YOLO11n_fecha_v1'/'weights'/'best.pt'
if not YAML.is_file():raise FileNotFoundError('Primero ejecuta notebook 02 (CPU) para generar el YAML.')
if not BEST.is_file():raise FileNotFoundError('Primero ejecuta notebook 03 (GPU) para generar best.pt.')
CFG=yaml.safe_load(YAML.read_text(encoding='utf8'))
TEST_TXT=Path(CFG['test']);assert TEST_TXT.is_file()
SAMPLES=[Path(x) for x in TEST_TXT.read_text(encoding='utf8').splitlines() if x.strip()]
assert len(SAMPLES)>0
model=YOLO(str(BEST))
OUT=RES/'02_Segmentacion'/'YOLO11n_fecha_v1'
OUT.mkdir(parents=True,exist_ok=True)
METR=RES/'03_Metricas'/'CERDOS_04_Evaluacion_test.json'
print('Imágenes en prueba:',len(SAMPLES));print('Modelo:',BEST)

In [ ]:
if METR.exists():
    prev=json.loads(METR.read_text(encoding='utf8'))
    if prev.get('weights_sha256') != hashlib.sha256(BEST.read_bytes()).hexdigest():
        raise RuntimeError('Cambió best.pt; la evaluación anterior no corresponde a estos pesos. Se requiere nueva versión del experimento.')
    print('✅ La evaluación en test ya existe; se reutiliza sin volver a consultar el test.')
    m=json.loads(METR.read_text(encoding='utf8'))
else:
    scores=model.val(data=str(YAML),split='test',device=0,imgsz=640,batch=8,plots=False,
                     project=str(OUT),name='metricas_test',exist_ok=True)
    m={'etapa':'04_evaluacion_GPU','fecha_utc':datetime.now(timezone.utc).isoformat(),
       'test_count':len(SAMPLES),'weights':str(BEST),'weights_sha256':hashlib.sha256(BEST.read_bytes()).hexdigest(),'split':'test_agrupado_por_fecha',
       'seg_map50':float(scores.seg.map50),'seg_map50_95':float(scores.seg.map),
       'seg_map75':float(scores.seg.map75),'box_map50':float(scores.box.map50),
       'box_map50_95':float(scores.box.map),
       'limitaciones':'Las fechas no garantizan independencia por granja o animal.'}
    METR.write_text(json.dumps(m,ensure_ascii=False,indent=2),encoding='utf8')
print('Resultados calculados (no inventados):')
print({k:v for k,v in m.items() if k.endswith(('map50','map50_95','map75'))})

In [ ]:
# Predicciones cualitativas de muestra: evita generar miles de PNG en Drive.
ids=np.unique(np.linspace(0,len(SAMPLES)-1,min(12,len(SAMPLES)),dtype=int))
rows=[]
for i in ids:
    img=SAMPLES[int(i)]
    rr=model.predict(source=str(img),device=0,conf=0.25,verbose=False,save=False)[0]
    annotated=rr.plot()
    imgout=OUT/f'{img.stem}_prediccion.png'
    cv2.imwrite(str(imgout),annotated)
    boxes=rr.boxes
    n=int(len(boxes)) if boxes is not None else 0
    confidencias=boxes.conf.detach().cpu().numpy().tolist() if n else []
    areas=[]
    if rr.masks is not None and rr.masks.data is not None:
        areas=rr.masks.data.detach().float().cpu().numpy().sum(axis=(1,2)).astype(float).tolist()
    rows.append({'imagen':img.name,'cerdos_detectados':n,
                 'confianza_media':float(np.mean(confidencias)) if confidencias else None,
                 'suma_areas_mascara_px2_aprox':float(sum(areas)) if areas else 0,
                 'archivo_png':imgout.name})

EXCEL=RES/'03_Metricas'/'CERDOS_04_Resultados_Segmentacion.xlsx'
with pd.ExcelWriter(EXCEL,engine='openpyxl') as wr:
    pd.DataFrame([m]).to_excel(wr,sheet_name='Metricas_test',index=False)
    pd.DataFrame(rows).to_excel(wr,sheet_name='Predicciones_ejemplo',index=False)
print('✅ Predicciones PNG:',OUT,'\n✅ Excel:',EXCEL)

In [ ]:
from google.colab import files
marca=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
ZIP=EXPO/f'CERDOS_04_Segmentacion_Resultados_{marca}.zip'
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for f in sorted(OUT.rglob('*')):
        if f.is_file():z.write(f,arcname=str(Path('segmentacion')/f.relative_to(OUT)))
    for f in (EXCEL,METR,DATASET_DIR/'split_fecha_v1'/'particion_fecha_manifest.json'):
        if f.is_file():z.write(f,arcname=str(Path('informes')/f.name))
with zipfile.ZipFile(ZIP) as z:
    if z.testzip():raise RuntimeError('ZIP generado con error.')
print('✅ ZIP de resultados:',ZIP)
print('✅ El ZIP no incluye el dataset original ni vuelve a entrenar.')
files.download(str(ZIP))

## Interpretación científica
Las métricas de máscaras corresponden a **segmentación de instancias**. La fiabilidad del seguimiento y de los indicadores conductuales se estudiará **más adelante con videos anotados**. No concluir generalización a otra granja a partir de una sola fuente.